In [1]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime
from zoneinfo import ZoneInfo

url = "https://raw.githubusercontent.com/vaish-vkb/Google-Play-Store-Analytics/main/Play%20Store%20Data.csv"

df6 = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Rows and columns:", df6.shape)
print(df6.columns.tolist())
display(df6.head())


Dataset loaded successfully!
Rows and columns: (10841, 13)
['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [2]:

# Convert installs into numbers
df6["Installs"] = (
    df6["Installs"].astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)
df6["Installs"] = pd.to_numeric(df6["Installs"], errors="coerce")

# Convert price into numbers
df6["Price"] = (
    df6["Price"].astype(str)
    .str.replace("$", "", regex=False)
)
df6["Price"] = pd.to_numeric(df6["Price"], errors="coerce")

# Convert size into MB
def size_to_mb(value):
    value = str(value).strip().upper()
    if value.endswith("M"):
        return pd.to_numeric(value[:-1], errors="coerce")
    elif value.endswith("K"):
        return pd.to_numeric(value[:-1], errors="coerce") / 1024
    return np.nan

df6["Size_MB"] = df6["Size"].apply(size_to_mb)

# Extract minimum Android version
def android_version(value):
    value = str(value).strip().lower()
    if value in ["nan", "varies with device"]:
        return np.nan
    version = value.replace(" and up", "").strip()
    try:
        return float(version)
    except ValueError:
        return np.nan

df6["Android_Version"] = df6["Android Ver"].apply(android_version)

# Estimated revenue: installs multiplied by app price
df6["Estimated_Revenue"] = df6["Installs"] * df6["Price"]

# Apply filters
filtered6 = df6[
    (df6["Installs"] >= 10000) &
    (df6["Estimated_Revenue"] >= 10000) &
    (df6["Android_Version"] > 4.0) &
    (df6["Size_MB"] > 15) &
    (df6["Content Rating"] == "Everyone") &
    (df6["App"].astype(str).str.len() <= 30)
].copy()

print("Rows after filtering:", len(filtered6))
print("App types:", filtered6["Type"].value_counts())
print("Categories:", filtered6["Category"].nunique())
display(filtered6[[
    "App", "Category", "Type", "Installs",
    "Price", "Estimated_Revenue", "Size_MB",
    "Android Ver", "Content Rating"
]].head(10))


Rows after filtering: 33
App types: Type
Paid    33
Name: count, dtype: int64
Categories: 13


,App,Category,Type,Installs,Price,Estimated_Revenue,Size_MB,Android Ver,Content Rating
853,Toca Life: City,EDUCATION,Paid,500000.0,3.99,1995000.0,24.0,4.4 and up,Everyone
854,Toca Life: Hospital,EDUCATION,Paid,100000.0,3.99,399000.0,24.0,4.4 and up,Everyone
1335,Meditation Studio,HEALTH_AND_FITNESS,Paid,10000.0,3.99,39900.0,29.0,4.3 and up,Everyone
1831,The Game of Life,GAME,Paid,100000.0,2.99,299000.0,63.0,4.4 and up,Everyone
1833,The Room: Old Sins,GAME,Paid,100000.0,4.99,499000.0,48.0,4.4 and up,Everyone
1839,Monument Valley 2,GAME,Paid,100000.0,4.99,499000.0,33.0,4.4 and up,Everyone
2151,Toca Life: City,FAMILY,Paid,500000.0,3.99,1995000.0,24.0,4.4 and up,Everyone
2293,Acupuncture Assistant,MEDICAL,Paid,10000.0,7.99,79900.0,39.0,4.1 and up,Everyone
2883,Facetune - For Free,PHOTOGRAPHY,Paid,1000000.0,5.99,5990000.0,48.0,4.1 and up,Everyone
2912,Facetune - For Free,PHOTOGRAPHY,Paid,1000000.0,5.99,5990000.0,48.0,4.1 and up,Everyone


In [3]:

# Find top 3 categories based on total installs
category_totals6 = (
    filtered6.groupby("Category", as_index=False)
    .agg(Total_Installs=("Installs", "sum"))
    .sort_values("Total_Installs", ascending=False)
)

top3_categories6 = category_totals6.head(3)["Category"].tolist()

print("Top 3 categories:", top3_categories6)

top3_data6 = filtered6[
    filtered6["Category"].isin(top3_categories6)
].copy()

display(top3_data6[[
    "App", "Category", "Type", "Installs", "Estimated_Revenue"
]])


Top 3 categories: ['PHOTOGRAPHY', 'FAMILY', 'PERSONALIZATION']


,App,Category,Type,Installs,Estimated_Revenue
2151,Toca Life: City,FAMILY,Paid,500000.0,1995000.0
2883,Facetune - For Free,PHOTOGRAPHY,Paid,1000000.0,5990000.0
2912,Facetune - For Free,PHOTOGRAPHY,Paid,1000000.0,5990000.0
2950,Facetune - For Free,PHOTOGRAPHY,Paid,1000000.0,5990000.0
3405,HD Widgets,PERSONALIZATION,Paid,1000000.0,990000.0
4260,Cut the Rope GOLD,FAMILY,Paid,1000000.0,990000.0
4429,Detect-O-Gromit (D.O.G 2),FAMILY,Paid,10000.0,24900.0
6919,HD Widgets,PERSONALIZATION,Paid,1000000.0,990000.0
7555,Swift Dark CM / CM13 Theme,PERSONALIZATION,Paid,10000.0,19900.0
7585,CN Superstar Soccer: Goal!!!,FAMILY,Paid,100000.0,299000.0


In [4]:

# Apply common filters first
base6 = df6[
    (df6["Installs"] >= 10000) &
    (df6["Android_Version"] > 4.0) &
    (df6["Size_MB"] > 15) &
    (df6["Content Rating"] == "Everyone") &
    (df6["App"].astype(str).str.len() <= 30) &
    (df6["Type"].isin(["Free", "Paid"]))
].copy()

# Estimate revenue
base6["Estimated_Revenue"] = base6["Installs"] * base6["Price"]

# Keep revenue >= $10,000 for paid apps; keep free apps for comparison
filtered6 = base6[
    (base6["Type"] == "Free") |
    (
        (base6["Type"] == "Paid") &
        (base6["Estimated_Revenue"] >= 10000)
    )
].copy()

# Recalculate top 3 categories using filtered installs
category_totals6 = (
    filtered6.groupby("Category", as_index=False)
    .agg(Total_Installs=("Installs", "sum"))
    .sort_values("Total_Installs", ascending=False)
)

top3_categories6 = category_totals6.head(3)["Category"].tolist()
top3_data6 = filtered6[
    filtered6["Category"].isin(top3_categories6)
].copy()

print("Top 3 categories:", top3_categories6)
print("\nFree vs Paid app counts:")
display(top3_data6.groupby(["Category", "Type"]).size().unstack(fill_value=0))


Top 3 categories: ['GAME', 'FAMILY', 'TOOLS']

Free vs Paid app counts:


Type,Free,Paid
Category,,
FAMILY,222,7
GAME,141,8
TOOLS,32,1


In [6]:

# Summarize average installs and average estimated revenue
summary6 = (
    top3_data6.groupby(["Category", "Type"], as_index=False)
    .agg(
        Average_Installs=("Installs", "mean"),
        Average_Revenue=("Estimated_Revenue", "mean")
    )
)

fig6 = go.Figure()

# Left axis: Average installs
for app_type in ["Free", "Paid"]:
    data = summary6[summary6["Type"] == app_type]

    fig6.add_trace(go.Bar(
        x=data["Category"],
        y=data["Average_Installs"],
        name=f"{app_type} - Average Installs",
        yaxis="y"
    ))

# Right axis: Average estimated revenue
for app_type in ["Free", "Paid"]:
    data = summary6[summary6["Type"] == app_type]

    fig6.add_trace(go.Scatter(
        x=data["Category"],
        y=data["Average_Revenue"],
        name=f"{app_type} - Average Estimated Revenue",
        yaxis="y2",
        mode="lines+markers+text",
        text=data["Type"],
        textposition="top center"
    ))

fig6.update_layout(
    title="Task 6: Average Installs vs Estimated Revenue — Free vs Paid Apps",
    xaxis_title="Top 3 App Categories",
    yaxis=dict(title="Average Installs"),
    yaxis2=dict(
        title="Average Estimated Revenue ($)",
        overlaying="y",
        side="right"
    ),
    barmode="group",
    template="plotly_white",
    height=650,
    legend_title="App Type and Metric"
)

# Temporarily display for testing

# Show chart only between 1 PM and 2 PM IST
now_ist = datetime.now(ZoneInfo("Asia/Kolkata"))
current_minutes = now_ist.hour * 60 + now_ist.minute

start_minutes = 13 * 60
end_minutes = 14 * 60

if start_minutes <= current_minutes < end_minutes:
    print("Task 6 chart is available now (1 PM - 2 PM IST).")
    fig6.show()
else:
    print("Task 6 chart is hidden. Available daily from 1 PM to 2 PM IST.")



Task 6 chart is hidden. Available daily from 1 PM to 2 PM IST.
